In [7]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_ETL_Pipeline") \
    .getOrCreate()

In [9]:
# 1. Membaca ketiga sumber data
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# 2. Tampilkan jumlah baris dan schema masing-masing
print("--- DATA TRANSAKSI ---")
print(f"Jumlah baris: {df_trx.count()}")
df_trx.printSchema()

print("--- DATA PRODUK ---")
print(f"Jumlah baris: {df_produk.count()}")
df_produk.printSchema()

print("--- DATA ULASAN ---")
print(f"Jumlah baris: {df_ulasan.count()}")
df_ulasan.printSchema()

--- DATA TRANSAKSI ---
Jumlah baris: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

--- DATA PRODUK ---
Jumlah baris: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

--- DATA ULASAN ---
Jumlah baris: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [10]:
# 1. Join transaksi ke ulasan (LEFT JOIN: karena tidak semua transaksi punya ulasan)
df_merged = df_trx.join(df_ulasan, on="order_id", how="left")

# 2. Join ke produk (INNER JOIN: karena setiap transaksi pasti punya produk valid)
df_merged = df_merged.join(df_produk, on="product_id", how="inner")

# 3. Tambahkan kolom total_pendapatan (unit_terjual x harga)
df_merged = df_merged.withColumn("total_pendapatan", F.col("unit_terjual") * F.col("harga"))

# Tampilkan sampel data hasil join
df_merged.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

In [11]:
# 1. Tambahkan kolom ada_ulasan SEBELUM mengisi nilai null pada rating
df_transformed = df_merged.withColumn("ada_ulasan", F.col("rating").isNotNull())

# 2. Isi nilai kosong (null) pada rating dengan angka 0
df_final = df_transformed.fillna({"rating": 0})

# Tampilkan data untuk verifikasi
df_final.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        10|     TX5|   

In [13]:
username = "adnan"

# Tambahkan hdfs://localhost:9000 atau hdfs://127.0.0.1:9000
hdfs_path = f"hdfs://localhost:9000/user/{username}/tugas6/hasil_etl"

# Simpan ke HDFS dalam format Parquet dipartisi berdasarkan kategori
df_final.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(hdfs_path)

# Verifikasi dengan membaca kembali data dari HDFS
df_read_back = spark.read.parquet(hdfs_path)
print(f"Jumlah baris terverifikasi di HDFS: {df_read_back.count()}")

Jumlah baris terverifikasi di HDFS: 5000


In [14]:
# Hitung persentase transaksi dengan ulasan (ada_ulasan = True) per kategori
df_insight = df_final.groupBy("kategori") \
    .agg(
        F.count("order_id").alias("total_transaksi"),
        F.sum(F.when(F.col("ada_ulasan") == True, 1).otherwise(0)).alias("jumlah_dengan_ulasan")
    ) \
    .withColumn("persentase_dengan_ulasan", (F.col("jumlah_dengan_ulasan") / F.col("total_transaksi")) * 100) \
    .orderBy("persentase_dengan_ulasan")

df_insight.show()

+------------+---------------+--------------------+------------------------+
|    kategori|total_transaksi|jumlah_dengan_ulasan|persentase_dengan_ulasan|
+------------+---------------+--------------------+------------------------+
|     Makanan|            536|                 369|       68.84328358208955|
|   Kesehatan|            961|                 662|       68.88657648283039|
|     Fashion|           1035|                 726|       70.14492753623188|
|Rumah Tangga|            815|                 575|        70.5521472392638|
|  Elektronik|           1653|                1168|         70.659407138536|
+------------+---------------+--------------------+------------------------+



Interpretasi Bisnis untuk Tim Marketing:
Kategori produk dengan persentase transaksi berulasan (ada_ulasan = True) paling rendah menunjukkan adanya potensi hambatan keterlibatan pelanggan (customer engagement) setelah pembelian. Informasi ini penting bagi tim marketing untuk merancang strategi retensi khusus, seperti memberikan insentif poin atau voucher diskon bagi pembeli di kategori tersebut agar mau memberikan ulasan, yang pada akhirnya membantu meningkatkan social proof dan kepercayaan calon pembeli baru.